# ImageNet-64 Preprocessing: Python Pickles → Memory-Mapped Binary

**Problem:** ImageNet-64 ships as Python pickle files (NumPy arrays). Deserializing these from Julia via PythonCall is painfully slow — ~3 minutes for 1.28M images. Doing this every epoch is unacceptable.

**Solution:** Run this notebook once to convert the pickles into a flat binary format that can be memory-mapped at training time. Each shard is written as raw `Float32` payloads with minimal headers (just ndims + shape). At training time, `MMapReader` maps these files and returns zero-copy array views in ~0.06 seconds.

**Output:** `build/train/` and `build/val/` directories containing:
- `count.ser` — number of shards
- `Xs.ser` — image data (64x64x3xN per shard, Float32)
- `ys.ser` — integer class labels
- `μs.ser` — per-shard channel means

In [ ]:
# ImageNet.load — unpickles Python data via PythonCall (slow, one-time cost)
# Utils.serialize — writes raw binary: [ndims][shape...][Float32 payload]
include("lib/imagenet.jl")
include("lib/utils.jl")

import .ImageNet
using .Utils

# Training Dataset

1.28M images across 10 pickle files (2 parts x 5 batches each). Each pickle is unpickled, permuted from NHWC to HWCN (Julia's column-major convention), and written as a flat binary shard.

In [ ]:
function load_train(files, output)
    function load_one(file::String)
        # Unpickle: returns (N, 64, 64, 3) from Python
        labels, images, μ = ImageNet.load(file, 64)
        # Permute to Julia convention: (64, 64, 3, N)
        pimages = permutedims(images, (2, 3, 4, 1))
        return labels, pimages, μ
    end
        
    base = "build/$output"
    ispath(base) && rm(base, force=true, recursive=true)
    mkpath(base)
    
    # Write shard count
	open("$base/count.ser", "w") do io
		write(io, length(files))
	end

    # Write each shard: images, means, and labels as flat binary
	ys = open("$base/ys.ser", "w")
	Xs = open("$base/Xs.ser", "w")
	μs = open("$base/μs.ser", "w")
	for (i, f) ∈ enumerate(files)
		y, X, μ = load_one(f)
		serialize(Xs, X)   # raw Float32 payload
		serialize(μs, μ)
		serialize(ys, y)

		h, w, c, n = size(X)
		@info "input" file=f n=n
	end
	close(ys)
	close(Xs)
	close(μs)
end

# ~3 minutes: the slow Python unpickling we only do once
@time begin
    load_train(vcat(
        readdir("$(ImageNet.IMAGENET_64)/Imagenet64_train_part1", join=true),
        readdir("$(ImageNet.IMAGENET_64)/Imagenet64_train_part2", join=true)
    ), "train")
end

# Validation Dataset

50k images in a single pickle file. Same conversion: unpickle → permute → write as flat binary.

In [ ]:
function load_val(file, output)
    # Unpickle: (N, 64, 64, 3) — no per-shard mean for validation
    labels, images, _ = ImageNet.load(file, 64; has_mean=false)
    # Permute to Julia convention: (64, 64, 3, N)
    pimages = permutedims(images, (2, 3, 4, 1))
        
    base = "build/$output"
    ispath(base) && rm(base, force=true, recursive=true)
    mkpath(base)
    
    # Single shard for validation set
	ys = open("$base/ys.ser", "w")
	Xs = open("$base/Xs.ser", "w")
    serialize(Xs, pimages)
    serialize(ys, labels)
	close(ys)
	close(Xs)

    h, w, c, n = size(pimages)
    @info "input" file=file n=n
end

@time load_val(
        readdir("$(ImageNet.IMAGENET_64)/Imagenet64_val", join=true)[1],
        "val")